# Excel 파일 로더

Microsoft Excel은 스프레드시트 데이터를 다루는 가장 인기 있는 애플리케이션입니다.

`.xlsx`와 `.xls` 형식의 Excel 파일을 LangChain Document로 변환하는 방법을 학습합니다.

## 주요 로더

1. **UnstructuredExcelLoader**: Excel 파일의 원시 텍스트 추출
2. **DataFrameLoader**: pandas DataFrame을 통한 구조화된 데이터 로드


## 환경 설정


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

# 샘플 Excel 파일 경로
FILE_PATH = "./data/titanic.xlsx"

# 데이터 파일 경로 확인
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {FILE_PATH}\n현재 디렉토리: {os.getcwd()}")

## 1. UnstructuredExcelLoader

`UnstructuredExcelLoader`는 Excel 파일을 로드하는 가장 편리한 방법입니다.

### 특징

- `.xlsx` 및 `.xls` 파일 모두 지원
- 원시 텍스트와 HTML 형식 모두 제공
- `mode="elements"`로 각 요소를 별도 문서로 추출 가능

### 필요한 패키지

```bash
pip install -qU langchain-community unstructured openpyxl msoffcrypto-tool
```

**참고**: `msoffcrypto-tool`은 암호화된 Excel 파일 처리에 필요합니다.


> 🎯 **강의 포인트**: Excel 로더는 두 가지 접근법이 있습니다. 단순 텍스트 추출은 `UnstructuredExcelLoader`, 특정 컬럼을 page_content로 활용하고 싶으면 pandas + `DataFrameLoader`를 사용하세요.

In [ ]:
from langchain_community.document_loaders import UnstructuredExcelLoader

# ============================================================
# TODO: UnstructuredExcelLoader로 Excel 파일을 로딩해보세요
# 힌트: UnstructuredExcelLoader(FILE_PATH, mode="elements")
# 예상 결과: 로드된 문서 개수와 첫 번째 문서 타입이 출력됩니다
# ============================================================

# 1단계: UnstructuredExcelLoader 생성
# - mode="elements": 각 요소(표, 텍스트 등)를 별도 Document로 추출
# - mode="single": 전체 내용을 하나의 Document로 합침 (기본값)
loader = UnstructuredExcelLoader(FILE_PATH, mode="elements")

# 2단계: 문서 로드
docs = loader.load()

print(f"로드된 문서 개수: {len(docs)}")
print(f"\n첫 번째 문서의 타입: {type(docs[0])}")

### page_content 확인

`page_content`에는 각 행의 데이터가 텍스트 형식으로 저장됩니다.


In [3]:
# 첫 번째 문서의 내용 출력 (처음 200자)
print("=" * 60)
print("📄 첫 번째 문서 내용:")
print("=" * 60)
print(docs[0].page_content[:200])


📄 첫 번째 문서 내용:
PassengerId Survived Pclass Name Sex Age SibSp Parch Ticket Fare Cabin Embarked 1 0 3 Braund, Mr. Owen Harris male 22 1 0 A/5 21171 7.25 S 2 1 1 Cumings, Mrs. John Bradley (Florence Briggs Thayer) fem


> 💡 **실무 팁**: `metadata["text_as_html"]`에 저장된 HTML 테이블은 LLM에게 직접 전달하면 표 구조를 이해하는 데 도움이 됩니다. 재무제표나 통계표처럼 표 내용 자체가 중요한 경우에 활용해보세요.

### metadata 확인

`metadata`의 `text_as_html` 키에는 데이터가 HTML 테이블 형식으로 저장됩니다.


In [4]:
# metadata의 text_as_html 출력 (처음 1000자)
print("=" * 60)
print("🔍 HTML 형식 메타데이터:")
print("=" * 60)
print(docs[0].metadata["text_as_html"][:1000])


🔍 HTML 형식 메타데이터:
<table><tr><td>PassengerId</td><td>Survived</td><td>Pclass</td><td>Name</td><td>Sex</td><td>Age</td><td>SibSp</td><td>Parch</td><td>Ticket</td><td>Fare</td><td>Cabin</td><td>Embarked</td></tr><tr><td>1</td><td>0</td><td>3</td><td>Braund, Mr. Owen Harris</td><td>male</td><td>22</td><td>1</td><td>0</td><td>A/5 21171</td><td>7.25</td><td/><td>S</td></tr><tr><td>2</td><td>1</td><td>1</td><td>Cumings, Mrs. John Bradley (Florence Briggs Thayer)</td><td>female</td><td>38</td><td>1</td><td>0</td><td>PC 17599</td><td>71.2833</td><td>C85</td><td>C</td></tr><tr><td>3</td><td>1</td><td>3</td><td>Heikkinen, Miss. Laina</td><td>female</td><td>26</td><td>0</td><td>0</td><td>STON/O2. 3101282</td><td>7.925</td><td/><td>S</td></tr><tr><td>4</td><td>1</td><td>1</td><td>Futrelle, Mrs. Jacques Heath (Lily May Peel)</td><td>female</td><td>35</td><td>1</td><td>0</td><td>113803</td><td>53.1</td><td>C123</td><td>S</td></tr><tr><td>5</td><td>0</td><td>3</td><td>Allen, Mr. William Henry</td><td>

## 2. DataFrameLoader

CSV와 마찬가지로 Excel 파일을 pandas DataFrame으로 읽은 후, `DataFrameLoader`를 사용하여 Document로 변환할 수 있습니다.

### 특징

- pandas의 강력한 데이터 처리 기능 활용
- 특정 컬럼을 `page_content`로 지정 가능
- 나머지 컬럼은 자동으로 `metadata`에 저장


> 🔑 **핵심 개념**: `DataFrameLoader`에서 `page_content_column`으로 지정하지 않은 나머지 컬럼들은 자동으로 `metadata`에 저장됩니다. RAG 필터링 조건으로 메타데이터를 활용할 수 있습니다.

In [5]:
import pandas as pd

# Excel 파일을 DataFrame으로 읽기
df = pd.read_excel(FILE_PATH)

print(f"DataFrame 크기: {df.shape}")
print(f"\n처음 3개 행:")
print(df.head(3))


DataFrame 크기: (891, 12)

처음 3개 행:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  


In [ ]:
from langchain_community.document_loaders import DataFrameLoader

# ============================================================
# TODO: DataFrameLoader로 DataFrame의 각 행을 Document로 변환해보세요
# 힌트: DataFrameLoader(df, page_content_column="Name")
# 예상 결과: 891개의 Document가 생성됩니다
# ============================================================

# 1단계: DataFrameLoader 생성
# - page_content_column: 이 컬럼의 값이 Document.page_content로 저장됨
# - 나머지 컬럼들은 모두 Document.metadata에 딕셔너리 형태로 저장됨
loader = DataFrameLoader(df, page_content_column="Name")

# 2단계: 문서 로드
docs = loader.load()

print(f"로드된 문서 개수: {len(docs)}")

### DataFrameLoader 결과 확인


In [7]:
# 첫 번째 문서의 내용과 메타데이터 출력
print("=" * 60)
print("📝 page_content (Name 컬럼):")
print("=" * 60)
print(docs[0].page_content)

print("\n" + "=" * 60)
print("🏷️  metadata (나머지 컬럼들):")
print("=" * 60)
print(docs[0].metadata)


📝 page_content (Name 컬럼):
Braund, Mr. Owen Harris

🏷️  metadata (나머지 컬럼들):
{'PassengerId': 1, 'Survived': 0, 'Pclass': 3, 'Sex': 'male', 'Age': 22.0, 'SibSp': 1, 'Parch': 0, 'Ticket': 'A/5 21171', 'Fare': 7.25, 'Cabin': nan, 'Embarked': 'S'}


## 💡 핵심 정리

### UnstructuredExcelLoader

```python
from langchain_community.document_loaders import UnstructuredExcelLoader

loader = UnstructuredExcelLoader("file.xlsx", mode="elements")
docs = loader.load()
```

- ✅ **장점**: 간단한 사용법, HTML 형식 제공
- ⚠️ **주의**: `msoffcrypto-tool` 패키지 필요

### DataFrameLoader

```python
import pandas as pd
from langchain_community.document_loaders import DataFrameLoader

df = pd.read_excel("file.xlsx")
loader = DataFrameLoader(df, page_content_column="Name")
docs = loader.load()
```

- ✅ **장점**: pandas의 강력한 데이터 처리, 컬럼 선택 자유
- ✅ **추천**: 구조화된 데이터 처리에 적합

### 선택 가이드

| 상황 | 추천 로더 |
|------|----------|
| 단순 텍스트 추출 | UnstructuredExcelLoader |
| 데이터 전처리 필요 | **DataFrameLoader** |
| HTML 형식 필요 | UnstructuredExcelLoader |
| 복잡한 데이터 조작 | **DataFrameLoader** |

**CSV vs Excel**: 단순 테이블 데이터라면 CSV가 더 간단하고, 여러 시트나 서식이 중요하면 Excel 로더 사용

---
